# Домашнее задание: Полноценная мультиагентная система с инструментами

Цель: спроектировать и реализовать систему из нескольких LLM‑агентов, которые координируются, обмениваются сообщениями и вызывают инструменты для решения реальной задачи end‑to‑end.


## Требования (обязательные)
- Роли агентов (минимум 3) с чётко разделёнными обязанностями и разными наборами инструментов для каждой роли.
- Коммуникация: шина сообщений или адресные сообщения (ask_agent / reply) с логами входящих/исходящих сообщений.
- Инструменты (минимум 3) с реальными побочными эффектами: например, работа с файлами/кодом/HTTP/данными/оценкой.
- SGR/Structured Output: схемы действий/сообщений (UseTool/AskAgent/Reply/Finish) с валидацией.
- Оркестрация: централизованный планировщик или децентрализованная логика (обосновать выбор).
- Демонстрация: один целевой сценарий, доведённый до finish, с логами и артефактами (код/отчёт/файлы).


## Оценивание (10 баллов)
- **3 балла — Дизайн агентов и протокола**
  - Чёткие роли и разграничение ответственности (1)
  - Разные доступные инструменты у разных ролей (1)
  - SGR/схемы и валидация structured output (1)
- **5 баллов — Мультиагентность и взаимодействие**
  - Координация и обмен сообщениями между агентами (2)
  - Реальные вызовы инструментов и побочные эффекты (2)
  - Обработка ошибок/ретраи/эскалация (1)
- **1 балл — Качество кода и отчёта**
  - Читаемость, структура, документация и воспроизводимость
- **1 балл — Бонус: >5 уникальных инструментов**
  - Разные по назначению функции, а не дубликаты


## Идеи задач
- Кодогенерация по спецификации: Planner → Coder → Tester → Reviewer.
- Data/RAG пайплайн: Ingestor → Indexer → Analyst → Evaluator.
- Веб‑интеграция: Researcher (web/http), Summarizer, Reporter, Verifier.

## Как сдавать
- Ноутбук с кодом агентов, инструментов и оркестратора.
- README с архитектурой (роли, взаимодействие), списком инструментов, схемами SGR, логами одного прогона, ограничениями и метриками.
- (Опционально) Короткое видео/гиф ≤3 мин с успешным прогоном.

## Штрафы
- − до 2: нет логов или невоспроизводимо
- − до 2: инструменты не делают реальных действий
- − до 1: смешение ролей/инструментов без обоснования


## Стартовые подсказки (не обязательно)
- Начните с 3 ролей и простого набора инструментов; затем добавляйте Reviewer/Verifier/Deployer.
- Схемы SGR: определите классы UseTool/AskAgent/Reply/Finish.
- Ограничьте инструменты по ролям на уровне оркестратора (enforcement).
- Логи: печатайте inbox/outbox, действие (JSON), результаты инструментов (stdout/error), diff при изменении файлов.
- Введите ретраи и сообщения «поясни ошибку», если JSON от модели невалиден или инструмент упал.


In [1]:
!pip install -q python-dotenv


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
from dotenv import load_dotenv

load_dotenv() 

True

In [3]:
# 1) Настройка OpenRouter (обязательное использование LLM)

import os
from typing import Optional

#os.environ["OPENAI_API_KEY"]="Your key here"

DEFAULT_MODEL = "qwen/qwen3-30b-a3b-instruct-2507"  # можно поменять на совместимую с JSON schema
openrouter_client = None

if os.environ.get("OPENAI_API_KEY"):
    try:
        from openai import OpenAI
        openrouter_client = OpenAI(
            base_url=os.environ["OPENAI_API_BASE"],
            api_key=os.environ["OPENAI_API_KEY"],
        )
        print("OpenRouter готов. Модель:", DEFAULT_MODEL)
    except Exception as e:
        raise RuntimeError(f"Не удалось инициализировать OpenRouter: {e}")
else:
    raise RuntimeError("OPENAI_API_KEY не найден. Установите ключ в окружении: export OPENROUTER_API_KEY=...")


OpenRouter готов. Модель: qwen/qwen3-30b-a3b-instruct-2507


In [4]:
# 2) вызов LLM с JSON Schema

from pydantic import BaseModel
from typing import Optional
import json

def call_llm_with_schema(
    prompt: str,
    response_schema: BaseModel,
    system_prompt: Optional[str] = None
) -> BaseModel:
    if openrouter_client is None:
        raise RuntimeError("OpenRouter клиент не инициализирован. Установите OPENROUTER_API_KEY и перезапустите kernel.")
    
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    
    schema_dict = response_schema.model_json_schema()
    resp = openrouter_client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=messages,
        response_format={
            "type": "json_schema",
            "json_schema": {"name": response_schema.__name__, "schema": schema_dict, "strict": True},
        },
        temperature=0.2,
        max_tokens=1500,
    )
    raw = resp.choices[0].message.content or ""
    
    def extract_json(s: str) -> str:
        s = s.strip()
        if s.startswith("```"):
            lines = s.splitlines()
            if lines and lines[0].startswith("```"):
                lines = lines[1:]
            if lines and lines[-1].strip().startswith("```"):
                lines = lines[:-1]
            s = "\n".join(lines)
        s = s.strip()
        try:
            json.loads(s)
            return s
        except Exception:
            pass
        first = s.find('{'); last = s.rfind('}')
        if first != -1 and last != -1 and last > first:
            cand = s[first:last+1]
            try:
                json.loads(cand)
                return cand
            except Exception:
                return s
        return s
    parsed = extract_json(raw)
    return response_schema.model_validate_json(parsed)


## Решаем задачу
- Кодогенерация по спецификации: Planner → Coder → Tester → Reviewer.

## 🏛 Обоснование архитектуры оркестрации

В проекте используется централизованная оркестрация через PlannerAgent вместо децентрализованной логики. Вот почему это является оптимальным решением:

### Преимущества централизованной оркестрации:

1. **Контроль потока выполнения:**
   - PlannerAgent как центральный координатор всех решений о маршрутизации
   - Все сообщения от рабочих агентов возвращаются к Planner
   - Анализ истории и контекста для принятия решений о следующем шаге

2. **Предсказуемость процесса:**
   - Четкая последовательность действий: Coder → Tester → Reviewer
   - Возможность контролируемых циклических возвратов
   - Единая точка принятия решений упрощает отладку

3. **Управление состоянием:**
   - Централизованное хранение истории взаимодействий в Bus
   - Полный контекст всех предыдущих действий
   - Эффективное отслеживание прогресса

4. **Разделение ответственности:**
   - Рабочие агенты фокусируются только на своих задачах
   - PlannerAgent отвечает за координацию
   - Четкое разграничение ролей

5. **Масштабируемость и поддержка:**
   - Простота добавления новых агентов
   - Изменения в маршрутизации требуют модификации только PlannerAgent
   - Легкость внедрения новых правил

### Почему не децентрализованная логика:

1. **Сложность координации:**
   - Усложнение логики каждого агента
   - Трудности в обеспечении согласованности действий
   - Больше точек принятия решений

2. **Риски конфликтов:**
   - Возможные конфликты решений между агентами
   - Сложности с обеспечением правильной последовательности
   - Риски зацикливания

3. **Сложность отладки:**
   - Труднее отследить причины проблем
   - Сложнее контролировать процесс
   - Больше точек отказа

4. **Дублирование логики:**
   - Каждому агенту нужна своя логика принятия решений
   - Увеличение кодовой базы
   - Больше возможностей для ошибок

5. **Контроль качества:**
   - Сложности с обеспечением единых стандартов
   - Менее предсказуемые результаты
   - Труднее внедрять изменения

In [5]:
from openai import OpenAI
import json

class OpenAILLM:
    def __init__(self, model: str, api_key: str, base_url: str = None):
        kwargs = {"api_key": api_key}
        if base_url:
            kwargs["base_url"] = base_url
        self.model = model
        self.client = OpenAI(**kwargs)


    
    def complete(self, messages, schema=None, **kwargs):
        # messages должен быть списком сообщений (чат-формат)
        if isinstance(messages, str):
            messages = [{"role": "user", "content": messages}]
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            response_format=kwargs.get("response_format", {"type": "json_object"}),
            max_tokens=kwargs.get("max_tokens", 1024),
            temperature=kwargs.get("temperature", 0)
        )
        out = resp.choices[0].message.content
        if schema is not None:
            try:
                data = json.loads(out)
            except Exception:
                raise RuntimeError(f"LLM did not return JSON: {out}")
            try:
                res = schema(**data)
            except Exception as e:
                raise RuntimeError(f"Rejected by {schema.__name__} schema: {data} / {e}")
            return res.dict()
        else:
            return out


In [6]:
from typing import Any, Dict, Optional, Union, Literal
from pydantic import BaseModel, Field

class AskAgent(BaseModel):
    action: Literal["ask_agent"] = "ask_agent"
    target: Literal["planner", "coder", "tester", "reviewer"]
    message: str

class UseTool(BaseModel):
    action: Literal["use_tool"] = "use_tool"
    tool_name: str
    args: Dict[str, Any] = Field(default_factory=dict)

class Reply(BaseModel):
    action: Literal["reply"] = "reply"
    content: str

class Finish(BaseModel):
    action: Literal["finish"] = "finish"
    summary: str

class AgentStep(BaseModel):
    step: Union[AskAgent, UseTool, Reply, Finish]

class BusMessage(BaseModel):
    sender: str
    recipient: str  # имя агента или "broadcast"
    content: str
    meta: Optional[Dict[str, Any]] = None


In [7]:
from typing import Optional, List, Any
import json

class BaseAgent:
    name: str = ""
    description: str = ""
    tools: List[str] = []

    def __init__(self, llm=None):
        self.llm = llm

    def decide(self, context: dict) -> AgentStep:
        raise NotImplementedError


def ensure_keys(d, keys, agent_name="agent"):
    missing = [k for k in keys if k not in d]
    if missing:
        raise ValueError(f"{agent_name}: отсутствуют ключи {missing} в ответе LLM: {d}\nПример верного JSON: {{'action': ..., 'tool_name': ..., 'args': ...}} или аналогичные.")

def ensure_not_null(out, agent_name="agent"):
    if out is None:
        raise ValueError(f"{agent_name}: LLM ответил пустым результатом! Проверь prompt и лимит токенов.")
    if not (isinstance(out, dict) or isinstance(out, list)):
        raise ValueError(f"{agent_name}: LLM ответил не dict и не list: {out}")

class PlannerAgent(BaseAgent):
    name = "planner"
    description = (
        "Координирует всю мультиагентную работу; анализирует историю и решает, кому дальше отправить baton и с каким заданием. Может возвращать задачи на доработку."
    )
    tools: List[str] = []

    def decide(self, context: dict) -> AgentStep:
        if not self.llm:
            raise RuntimeError("LLM is required for this agent!")
        messages = [
            {
                "role": "system",
                "content": (
                    "Ты — PlannerAgent, менеджер мультиагентной системы. Твоя задача — после КАЖДОГО шага анализировать историю (history) и определять, кому и что делать дальше.\n"
                    "Ты всегда формируешь только ОДИН валидный JSON-ответ: либо {\"action\": \"ask_agent\", \"target\": <агент>, \"message\": <подробное задание>}, либо — если ВСЁ ГОТОВО — {\"action\": \"finish\", \"summary\": <кратко результат>}\n"
                    "ПРАВИЛА:\n"
                    "- Если результат РЕВЬЮЕРА — ok (Стиль в порядке/стиль кода в порядке), baton -> finish: action='finish', summary='Все проверки пройдены, пайплайн завершён.'\n"
                    "- Если результат ревьюера — ошибка стиля (issues/ошибка), baton -> coder, message='Стиль не пройден, исправь стиль.'\n"
                    "- Если результат run_tests — ok (status=ok, все тесты прошли), baton -> reviewer, message='Проверь стиль кода (lint_code)'.\n"
                    "- Если результат run_tests — ошибка (status!=ok), baton -> coder, message='Тесты не пройдены, доработай решение, затем сохрани и запусти тесты повторно.'\n"
                    "- Если последний шаг tester был успешно store_code (test_solution.py) — теперь baton -> tester, message='Выполни инструмент run_tests (solution.py, test_solution.py)' (не повторяй store_code подряд).\n"
                    "- Если последний шаг coder — store_code, baton -> tester, message='Сгенерируй юнит-тесты на solution.py, сохрани их как test_solution.py, без комментариев и markdown.'\n"
                    "- Baton всегда можешь вернуть любому агента при ошибке или для доработки.\n"
                    "- Message всегда четко: либо сохрани, либо запусти, либо исправь, либо заверши.\n"
                )
            },
            {
                "role": "user",
                "content": (
                    f"Задача: {context['task']}\nИстория действий: {context['history']}"
                )
            }
        ]
        out = self.llm.complete(messages=messages)
        if isinstance(out, str):
            out = json.loads(out)
        ensure_not_null(out, "PlannerAgent")
        # finish или ask_agent (выбираем по action)
        if out.get("action") == "finish":
            step = Finish(action="finish", summary=out.get("summary", "Готово"))
            return AgentStep(step=step)
        ensure_keys(out, ["action", "target", "message"], "PlannerAgent")
        step = AskAgent(**out)
        return AgentStep(step=step)

class CoderAgent(BaseAgent):
    name = "coder"
    description = (
        "Пишет код решения задачи. Сохраняет только через инструмент store_code. "
        "Может использовать read_code, lint_code, run_python."
    )
    tools: List[str] = ["store_code", "read_code", "run_python", "lint_code"]

    def decide(self, context: dict):
        if not self.llm:
            raise RuntimeError("LLM is required for this agent!")
        inst = "Реализуй is_prime."
        for msg in reversed(context.get("history", [])):
            if msg.get("recipient") == "coder" and msg.get("action") in ("ask_agent", "message"):
                inst = msg.get("content") or msg.get("message", inst)
                break
        messages = [
            {
                "role": "system",
                "content": (
                    "Ты CoderAgent. Ответ ВСЕГДА должен быть СТРОГО ОДНИМ JSON-объектом: {\"action\": \"use_tool\", \"tool_name\": \"store_code\", \"args\": {\"filename\": \"solution.py\", \"code\": \"<CODE>\"}}. "
                    "Никаких массивов, никаких других полей, только эти ключи!"
                )
            },
            {
                "role": "user",
                "content": f"{inst}\nЗадача: {context['task']}"
            }
        ]
        out = self.llm.complete(messages=messages)
        if isinstance(out, str):
            out = json.loads(out)
        ensure_not_null(out, "CoderAgent")
        ensure_keys(out, ["action", "tool_name", "args"], "CoderAgent:use_tool")
        return AgentStep(step=UseTool(**out))

class TesterAgent(BaseAgent):
    name = "tester"
    description = (
        "Пишет и запускает юнит-тесты к решению. Сохраняет тесты через store_code и запускает через run_tests."
    )
    tools: List[str] = ["store_code", "read_code", "run_tests"]

    def decide(self, context: dict):
        if not self.llm:
            raise RuntimeError("LLM is required for this agent!")
        # Детект задачи от PlannerAgent: если нужно запустить тесты — делаем run_tests
        last_message = ""
        history = context.get("history", [])
        # ищем последнее message для tester из истории
        for entry in reversed(history):
            if entry.get("recipient") == "tester":
                last_message = entry.get("content", "")
                break
        if any(key in last_message.lower() for key in ["run_tests", "запусти тест", "запусти тесты", "test_solution.py, чтобы убедиться", "выполни инструмент run_tests"]):
            return AgentStep(step=UseTool(action="use_tool", tool_name="run_tests", args={
                "filename": "solution.py",
                "test_file": "test_solution.py"
            }))
        # иначе стандартное поведение (генерация тестов)
        messages = [
            {
                "role": "system",
                "content": (
                    "Ты TesterAgent. Если тесты ещё не созданы (нет файла 'test_solution.py'), ответь СТРОГО ОДНИМ JSON-объектом: {\"action\": \"use_tool\", \"tool_name\": \"store_code\", \"args\": {\"filename\": \"test_solution.py\", \"code\": \"<TESTS>\"}}. "
                    "Если тесты уже созданы — обязательно вызови run_tests посредством: {\"action\": \"use_tool\", \"tool_name\": \"run_tests\", \"args\": {\"filename\": \"solution.py\", \"test_file\": \"test_solution.py\"}}. "
                    "Никаких массивов, никаких других полей, только эти ключи!"
                )
            },
            {
                "role": "user",
                "content": (
                    "Задача: %s" % context["task"]
                )
            }
        ]
        out = self.llm.complete(messages=messages)
        if isinstance(out, str):
            out = json.loads(out)
        ensure_not_null(out, "TesterAgent")
        ensure_keys(out, ["action", "tool_name", "args"], "TesterAgent:use_tool")
        return AgentStep(step=UseTool(**out))

class ReviewerAgent(BaseAgent):
    name = "reviewer"
    description = (
        "Проверяет стиль, качество и эффективность кода, использует только инструменты read_code, lint_code, summarize_text."
    )
    tools: List[str] = ["read_code", "lint_code", "summarize_text"]

    def decide(self, context: dict):
        if not self.llm:
            raise RuntimeError("LLM is required for this agent!")
        messages = [
            {
                "role": "system",
                "content": (
                    "Ты ReviewerAgent. Ответ ВСЕГДА должен быть СТРОГО ОДНИМ JSON-объектом: {\"action\": \"use_tool\", \"tool_name\": \"lint_code\", \"args\": {\"code\": \"<CODE_TO_CHECK>\"}}. "
                    "Никаких массивов, никаких других полей!"
                )
            },
            {
                "role": "user",
                "content": (
                    f"Проверь стиль решения в solution.py. Задача: {context['task']}"
                )
            }
        ]
        out = self.llm.complete(messages=messages)
        if isinstance(out, str):
            out = json.loads(out)
        ensure_not_null(out, "ReviewerAgent")
        ensure_keys(out, ["action", "tool_name", "args"], "ReviewerAgent:use_tool")
        return AgentStep(step=UseTool(**out))

def get_agents(llm=None):
    return [
        PlannerAgent(llm=llm),
        CoderAgent(llm=llm),
        TesterAgent(llm=llm),
        ReviewerAgent(llm=llm)
    ]


In [8]:
from collections import deque

class Bus:
    """
    Message bus: публикует и отдаёт только объекты BusMessage
    """
    def __init__(self):
        self._messages = deque()  # только BusMessage
        self._history = []

    def publish(self, message: BusMessage):
        """
        Положить строго BusMessage в очередь. Только BusMessage, иначе ошибка.
        """
        if not isinstance(message, BusMessage):
            raise TypeError("Bus.publish принимает только BusMessage!")
        self._messages.append(message)
        self._history.append(message)

    def get_next_for(self, agent_name: str):
        """
        Забрать следующее сообщение для агента. Возвращает BusMessage или None.
        """
        for idx, msg in enumerate(self._messages):
            if msg.recipient == agent_name:
                found = msg
                self._messages = deque([m for i, m in enumerate(self._messages) if i != idx])
                return found
        return None

    def get_history(self):
        return list(self._history)

    def is_empty(self):
        return not self._messages


In [9]:
class Workspace:
    def __init__(self):
        self.files = {}

    def store_code(self, filename: str, code: str):
        self.files[filename] = code
        return {"message": f"stored {filename}", "filename": filename, "chars": len(code)}

    def read_code(self, filename: str):
        return {"filename": filename, "code": self.files.get(filename, "")}


In [10]:
from typing import Dict, Any

class Tools:
    """
    Унифицированный набор инструментов для взаимодействия агентов
    с рабочим пространством в object-oriented стиле.
    """
    def __init__(self, ws: Workspace = None):
        self.ws = ws or Workspace()

    def call(self, tool_name: str, args: Dict[str, Any]) -> Any:
        args = args or {}
        if tool_name == "store_code":
            filename = args.get("filename", "solution.py")
            code = args.get("code") or args.get("content") or ""
            self.ws.store_code(filename, code)
            return {"status": "ok", "message": f"Код для {filename} сохранён."}

        if tool_name == "read_code":
            filename = args.get("filename", "solution.py")
            code = self.ws.read_code(filename)["code"]
            return {"status": "ok", "code": code}

        if tool_name == "run_python":
            from io import StringIO
            import sys, traceback
            import builtins
            buf = StringIO()
            old_stdout = sys.stdout
            sys.stdout = buf
            try:
                exec(
                    args.get("code", ""),
                    {"__builtins__": {
                        "print": print,
                        "range": range,
                        "len": len,
                        "int": int,
                        "float": float,
                        "str": str,
                        "bool": bool,
                        "list": list,
                        "dict": dict,
                        "set": set,
                        "type": type,
                        "object": object,
                        "globals": globals,
                        "isinstance": isinstance,
                        "issubclass": issubclass,
                        "__name__": "__main__",
                        "__import__": __import__,
                        "__build_class__": builtins.__build_class__,
                    }}
                )
                output = buf.getvalue()
                return {"status": "ok", "output": output}
            except Exception:
                err = "Ошибка:\n" + traceback.format_exc()
                return {"status": "error", "error": err}
            finally:
                sys.stdout = old_stdout

        if tool_name == "run_tests":
            filename = args.get("filename", "solution.py")
            test_file = args.get("test_file", "test_solution.py")
            code = self.ws.read_code(filename)["code"]
            tests = self.ws.read_code(test_file)["code"]
            if not code:
                return {"status": "error", "error": f"Нет файла {filename}"}
            if not tests:
                return {"status": "error", "error": f"Нет тестов в {test_file}"}
            # Удалить импорт решения из тестов (универсально)
            import re
            tests_clean = re.sub(r'^\s*from solution import .*$', '', tests, flags=re.MULTILINE)
            tests_clean = re.sub(r'^\s*import solution\s*$', '', tests_clean, flags=re.MULTILINE)
            # Удалить/заменить unittest.main()
            tests_clean = re.sub(
                r'^if __name__ == [\'"]__main__[\'"]:(.|\n)*?unittest\.main\(\)',
                '', tests_clean, flags=re.MULTILINE)
            # Добавить явный запуск вручную
            custom_test_runner = '''\nimport unittest\nimport sys\ndef _run_all_tests():\n    suite = unittest.TestSuite()\n    for obj in list(globals().values()):\n        if isinstance(obj, type) and issubclass(obj, unittest.TestCase):\n            suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(obj))\n    runner = unittest.TextTestRunner(stream=sys.stdout, verbosity=2)\n    result = runner.run(suite)\n_run_all_tests()\n'''
            full_code = code + "\n" + tests_clean + "\n" + custom_test_runner
            result = self.call("run_python", {"code": full_code})
            if result["status"] == "error":
                return {"status": "error", "error": result["error"]}
            return {"status": "ok", "message": "Все тесты пройдены!", "output": result["output"]}

        if tool_name == "lint_code":
            code = args.get("code", "")
            if not code:
                return {"status": "error", "error": "Не передан параметр code!"}
            if "    " in code and "\t" in code:
                return {"status": "error", "error": "Смешаны табы и пробелы!"}
            return {"status": "ok", "message": "Стиль кода в порядке."}

        if tool_name == "summarize_text":
            text = args.get("text", "")
            return {"status": "ok", "summary": text[:80] + ("..." if len(text) > 80 else "")}

        return {"status": "error", "error": f"Неизвестный инструмент '{tool_name}'"}

# Глобальный workspace и объект Tools — для совместимости main.py:
ws = Workspace()
tools_obj = Tools(ws)

def get_tools() -> Tools:
    return tools_obj

In [11]:


class Orchestrator:
    def __init__(self, agents, tools, task):
        self.tools = tools
        self.task = task
        self.bus = Bus()
        self.agent_map = {agent.name: agent for agent in agents}
        self.planner = self.agent_map['planner']

    def run(self):
        # Initial baton: manager → planner (начни работу, задача)
        self.bus.publish(BusMessage(
            sender="manager",
            recipient="planner",
            content=self.task,
            meta={"action": "ask_agent"}
        ))

        while not self.bus.is_empty():
            msg = self.bus._messages.popleft()
            print(f"\n[TRACE] Baton у {msg.recipient}: {msg.meta} :: content={msg.content}")
            print(f"[TRACE] Длина истории: {len(self.bus.get_history())}")
            context = {
                "task": self.task,
                "history": [h.dict() for h in self.bus.get_history()]
            }
            # 1. Если baton у planner — только Planner решает, что дальше
            if msg.recipient == "planner":
                print("[TRACE] Planner думает… (вызов LLM)")
                plan_step = self.planner.decide(context)
                # Если это finish — завершаем сценарий
                if hasattr(plan_step.step, "action") and plan_step.step.action == "finish":
                    finish_msg = BusMessage(
                        sender="planner",
                        recipient="manager",
                        content=getattr(plan_step.step, "summary", "Завершено!"),
                        meta={"action": "finish"},
                    )
                    self.bus.publish(finish_msg)
                    print("\n=== Process finished! (Planner дал finish) ===\n")
                    break
                assert isinstance(plan_step.step, AskAgent)
                ask = plan_step.step
                print(f"[TRACE] Planner выбрал: {ask.target}, message={ask.message}")
                baton_msg = BusMessage(
                    sender="planner",
                    recipient=ask.target,
                    content=ask.message,
                    meta={"action": "ask_agent"}
                )
                self.bus.publish(baton_msg)
                continue

            # 2. Если baton у coder/tester/reviewer — действие в их стиле
            agent = self.agent_map.get(msg.recipient)
            if not agent:
                print(f"[ОШИБКА] Не найден агент для {msg.recipient}")
                break
            print(f"[TRACE] Агент {msg.recipient}: выполняет decide…")
            context["history"] = [h.dict() for h in self.bus.get_history()]
            step = agent.decide(context).step

            # Если это use_tool — реально выполнить инструмент
            if hasattr(step, 'action') and step.action == 'use_tool':
                print(f"[TRACE] Агент {msg.recipient}: вызывает инструмент {step.tool_name}")
                tool_result = self.tools.call(step.tool_name, step.args)
                print(f"[TRACE] Результат инструмента: {tool_result}")
                tool_msg = BusMessage(
                    sender=msg.recipient,
                    recipient="planner",  # baton назад к planner
                    content=str(tool_result),
                    meta={"action": "use_tool", "tool": step.tool_name, "args": step.args}
                )
                self.bus.publish(tool_msg)
            elif hasattr(step, 'action') and step.action == 'finish':
                finish_msg = BusMessage(
                    sender=msg.recipient,
                    recipient="manager",
                    content=getattr(step, 'summary', 'Завершено'),
                    meta={"action": "finish"}
                )
                self.bus.publish(finish_msg)
                print("\n=== Process finished! ===\n")
                break
            else:
                print(f"[TRACE] Агент {msg.recipient}: возвращает reply/meta={getattr(step, 'action', 'reply')}")
                self.bus.publish(BusMessage(
                    sender=msg.recipient,
                    recipient="planner",
                    content=str(getattr(step, 'content', '')),
                    meta={"action": getattr(step, 'action', 'reply')}
                ))
        return {
            "history": [m.dict() for m in self.bus.get_history()]
        }


In [12]:
import os
from dotenv import load_dotenv

load_dotenv()

def print_history(history):
    print("\n[Лог]")
    for step in history:
        sender = step.get('sender', '<NONE>')
        recipient = step.get('recipient', '<NONE>')
        action = step.get('meta', {}).get('action') or step.get('action', '')
        content = step.get('content', '')
        print(f"{sender} -> {recipient}: {action} {content}\n")

def main():
    api_key = os.getenv("OPENAI_API_KEY")
    base_url = os.getenv("OPENAI_API_BASE") or None
    llm = OpenAILLM(model="gpt-4o", api_key=api_key, base_url=base_url)
    agents = get_agents(llm=llm)
    tools = get_tools()
    task = "Реализовать функцию is_prime(n: int) -> bool, которая проверяет, является ли число простым."
    orchestrator = Orchestrator(agents=agents, tools=tools, task=task)
    context = orchestrator.run()

    print("\n[Файлы в рабочем пространстве]:")
    for fname, content in ws.files.items():
        print(f"\n--- {fname} ---\n{content}")

    print_history(context["history"])

if __name__ == "__main__":
    main()




[TRACE] Baton у planner: {'action': 'ask_agent'} :: content=Реализовать функцию is_prime(n: int) -> bool, которая проверяет, является ли число простым.
[TRACE] Длина истории: 1
[TRACE] Planner думает… (вызов LLM)


/var/folders/mn/74b9l1w97c94cgd97_5z_3n40000gp/T/ipykernel_45372/1295806384.py:24: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  "history": [h.dict() for h in self.bus.get_history()]


[TRACE] Planner выбрал: coder, message=Реализуй функцию is_prime(n: int) -> bool, которая проверяет, является ли число простым, и сохрани её как solution.py.

[TRACE] Baton у coder: {'action': 'ask_agent'} :: content=Реализуй функцию is_prime(n: int) -> bool, которая проверяет, является ли число простым, и сохрани её как solution.py.
[TRACE] Длина истории: 2
[TRACE] Агент coder: выполняет decide…


/var/folders/mn/74b9l1w97c94cgd97_5z_3n40000gp/T/ipykernel_45372/1295806384.py:59: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  context["history"] = [h.dict() for h in self.bus.get_history()]


[TRACE] Агент coder: вызывает инструмент store_code
[TRACE] Результат инструмента: {'status': 'ok', 'message': 'Код для solution.py сохранён.'}

[TRACE] Baton у planner: {'action': 'use_tool', 'tool': 'store_code', 'args': {'filename': 'solution.py', 'code': 'def is_prime(n: int) -> bool:\n    if n <= 1:\n        return False\n    if n <= 3:\n        return True\n    if n % 2 == 0 or n % 3 == 0:\n        return False\n    i = 5\n    while i * i <= n:\n        if n % i == 0 or n % (i + 2) == 0:\n            return False\n        i += 6\n    return True'}} :: content={'status': 'ok', 'message': 'Код для solution.py сохранён.'}
[TRACE] Длина истории: 3
[TRACE] Planner думает… (вызов LLM)
[TRACE] Planner выбрал: tester, message=Сгенерируй юнит-тесты на solution.py, сохрани их как test_solution.py, без комментариев и markdown.

[TRACE] Baton у tester: {'action': 'ask_agent'} :: content=Сгенерируй юнит-тесты на solution.py, сохрани их как test_solution.py, без комментариев и markdown.
[TRACE

/var/folders/mn/74b9l1w97c94cgd97_5z_3n40000gp/T/ipykernel_45372/1295806384.py:24: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  "history": [h.dict() for h in self.bus.get_history()]


[TRACE] Planner выбрал: reviewer, message=Проверь стиль кода (lint_code).

[TRACE] Baton у reviewer: {'action': 'ask_agent'} :: content=Проверь стиль кода (lint_code).
[TRACE] Длина истории: 8
[TRACE] Агент reviewer: выполняет decide…


/var/folders/mn/74b9l1w97c94cgd97_5z_3n40000gp/T/ipykernel_45372/1295806384.py:59: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  context["history"] = [h.dict() for h in self.bus.get_history()]


[TRACE] Агент reviewer: вызывает инструмент lint_code
[TRACE] Результат инструмента: {'status': 'ok', 'message': 'Стиль кода в порядке.'}

[TRACE] Baton у planner: {'action': 'use_tool', 'tool': 'lint_code', 'args': {'code': 'def is_prime(n: int) -> bool:\n    if n <= 1:\n        return False\n    if n <= 3:\n        return True\n    if n % 2 == 0 or n % 3 == 0:\n        return False\n    i = 5\n    while i * i <= n:\n        if n % i == 0 or n % (i + 2) == 0:\n            return False\n        i += 6\n    return True'}} :: content={'status': 'ok', 'message': 'Стиль кода в порядке.'}
[TRACE] Длина истории: 9
[TRACE] Planner думает… (вызов LLM)

=== Process finished! (Planner дал finish) ===


[Файлы в рабочем пространстве]:

--- solution.py ---
def is_prime(n: int) -> bool:
    if n <= 1:
        return False
    if n <= 3:
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            ret

/var/folders/mn/74b9l1w97c94cgd97_5z_3n40000gp/T/ipykernel_45372/1295806384.py:93: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  "history": [m.dict() for m in self.bus.get_history()]
